In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

In [ ]:
# --- Data Preparation ---
if not Path ("input.txt").exists():
    import urllib.request
    print("downloading")
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, "input.txt")
text = open("input.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)} #String to Index
itos = {i: ch for ch, i in stoi.items()} #Index to String
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

downloading


In [ ]:
class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size #How many characters the model look at once.

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # x and y are both sequences of length block_size
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

In [4]:
# --- Transformer Component ---
class Head(nn.Module):
    """ One head of masked self-attention """
    def __init__(self, emb_dim, head_size, block_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        v = self.value(x) # (B, T, head_size)

        # Compute attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5) # (B, T, T)
        # Causal mask: prevent tokens from looking into the future
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        # Perform the weighted aggregation of values
        out = wei @ v # (B, T, head_size)
        return out

In [5]:
class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention running in parallel """
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        head_size = emb_dim // num_heads
        self.heads = nn.ModuleList([Head(emb_dim, head_size, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Concatenate outputs from all heads along the channel/embedding dimension
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out) # Projection back into the residual path
        out = self.dropout(out)
        return out

class FeedForward(nn.Module):
    """ A simple linear layer followed by a non-linearity """
    def __init__(self, emb_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communicates (attention) then computes (feedforward) """
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(emb_dim)
        self.sa = MultiHeadAttention(emb_dim, num_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(emb_dim)
        self.ffwd = FeedForward(emb_dim, dropout)

    def forward(self, x):
        # Pre-LN combined with Residual connections
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [11]:
# -- TinyGPT ---
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=256, num_heads=8, num_layers=8, dropout=0.1):  #Increased Capacity to increase long-term sentences
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.blocks = nn.Sequential(*[
            Block(emb_dim, num_heads, block_size, dropout) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)

        tok_emb = self.token_embedding(x)      # (B, T, emb_dim)
        pos_emb = self.position_embedding(pos) # (T, emb_dim)

        h = tok_emb + pos_emb                  # (B, T, emb_dim)
        h = self.blocks(h)                     # (B, T, emb_dim)
        h = self.ln_f(h)                       # (B, T, emb_dim)
        logits = self.lm_head(h)               # (B, T, vocab_size)
        return logits

In [13]:
# --- Training Loop + Evaluation ---

def sequence_cross_entropy(logits, targets):
    # Reshape targets to (B*T) and logits to (B*T, V) for standard cross entropy
    return F.cross_entropy(logits.transpose(1, 2), targets)

def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)

        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)

        if max_steps is not None and step + 1 >= max_steps:
            break

    return total_loss / total_count

In [14]:
# --- Sampling and Text Generation ---

@torch.no_grad()
def sample_gpt(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=200):
    model.eval()
    # Initialize a blank context window filled with 0s
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)

    # Pre-fill context with your prompt characters
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)

    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :] # Crop logits to focus only on the last step prediction
        probs = F.softmax(logits, dim=-1)

        # Sample using multinomial distribution instead of argmax for varied generation
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])

        # Shift the context window over by 1 to make room for the new token
        context = torch.cat([context[:, 1:], ix], dim=1)

    return "".join(out)

In [15]:
# --- Execution ---

if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # Hyperparameters
    block_size = 64
    batch_size = 64
    epochs = 100         # Adjust this up to 50 or 100 once you verify it works!
    max_steps_per_epoch = 150

    # Dataset setups
    dataset = NextTokenDataset(data, block_size)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Model instantiation
    model = TinyGPT(vocab_size, block_size).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

    print("Starting training script...")
    for epoch in range(epochs):
        loss = train_one_epoch(model, loader, optimizer, device, max_steps=max_steps_per_epoch)
        print(f"Epoch {epoch:2d} | Train Loss {loss:.4f}")

    print("\n--- GENERATED TEXT TEST ---")
    generated_sample = sample_gpt(model, block_size, stoi, itos, device, start_text="ROMEO:\n", max_new_tokens=300)
    print(generated_sample)

Using device: cuda
Starting training script...
Epoch  0 | Train Loss 2.5943
Epoch  1 | Train Loss 2.2332
Epoch  2 | Train Loss 2.0232
Epoch  3 | Train Loss 1.8807
Epoch  4 | Train Loss 1.7823
Epoch  5 | Train Loss 1.7118
Epoch  6 | Train Loss 1.6557
Epoch  7 | Train Loss 1.6156
Epoch  8 | Train Loss 1.5794
Epoch  9 | Train Loss 1.5520
Epoch 10 | Train Loss 1.5258
Epoch 11 | Train Loss 1.5082
Epoch 12 | Train Loss 1.4862
Epoch 13 | Train Loss 1.4732
Epoch 14 | Train Loss 1.4611
Epoch 15 | Train Loss 1.4403
Epoch 16 | Train Loss 1.4384
Epoch 17 | Train Loss 1.4219
Epoch 18 | Train Loss 1.4102
Epoch 19 | Train Loss 1.4031
Epoch 20 | Train Loss 1.3933
Epoch 21 | Train Loss 1.3846
Epoch 22 | Train Loss 1.3736
Epoch 23 | Train Loss 1.3730
Epoch 24 | Train Loss 1.3615
Epoch 25 | Train Loss 1.3551
Epoch 26 | Train Loss 1.3482
Epoch 27 | Train Loss 1.3417
Epoch 28 | Train Loss 1.3340
Epoch 29 | Train Loss 1.3326
Epoch 30 | Train Loss 1.3276
Epoch 31 | Train Loss 1.3170
Epoch 32 | Train Loss 1.3